# テーマ5 ニューラルネットワークの基礎

## 目的
- ニューラルネットワークを「多数の重みを使って入力と出力の関係を学習するモデル」として理解する
- 入力層、隠れ層、出力層、損失関数、エポックの役割に慣れる
- Breast Cancer データセットで小さなニューラルネットワーク分類を行う
- 学習曲線から、学習の様子や過学習の兆候を読み取る

## 今日の成果物
- モデルの入力と出力の形を確認する
- epoch ごとの loss と accuracy を可視化する
- 訓練データとテストデータの差から、過学習の兆候を考察する

## 注意
- このテンプレートは PyTorch を使います
- 環境に PyTorch がない場合は、教員側で事前セットアップが必要です


## ニューラルネットワークの考え方

ニューラルネットワークは、入力を何段階かの変換に通して、最終的な予測を出すモデルです。

1つの層では、入力 $x$ に重み $W$ とバイアス $b$ を使って次のような計算をします。

$$
z = Wx + b
$$

そのあと、ReLU などの活性化関数を通します。

$$
ReLU(z) = \max(0, z)
$$

この「線形変換」と「非線形変換」を組み合わせることで、単純な直線だけでは表しにくい関係も学習できるようになります。


### Step 0: ライブラリを読み込む
PyTorch とデータ準備用のライブラリを読み込みます。乱数を固定して、実行ごとの結果のぶれを小さくします。


In [ ]:
# ===== Step 0: ライブラリ読み込み =====
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device)


### Step 1: データを読み込む
Breast Cancer データセットを読み込みます。ニューラルネットワークに入れる前に、入力特徴量と正解ラベルを分けます。


In [ ]:
# ===== Step 1: データ読み込み =====
cancer = load_breast_cancer(as_frame=True)
df = cancer.frame.copy()

X = df[cancer.feature_names]
y = df['target']

print('入力の形 =', X.shape)
print('クラス名 =', list(cancer.target_names))
display(df.head())


### Step 2: 学習用データを作る
学習用とテスト用に分け、特徴量を標準化してから PyTorch の Tensor に変換します。表形式データは、画像と違って2次元の入力として扱います。


In [ ]:
# ===== Step 2: 学習データ作成 =====
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.long)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=64)

print('入力特徴量の数 =', X_train_tensor.shape[1])
print('train batch 数 =', len(train_loader))


### Step 3: ニューラルネットワークを定義する
入力層、隠れ層、出力層を持つ小さなモデルを作ります。出力は2クラス分のスコアになり、損失関数で正解との差を計算します。


#### 解説: 出力と損失関数

このモデルの最後の層は、2クラス分のスコアを出します。スコアそのものは確率ではありませんが、`CrossEntropyLoss` は内部で softmax を使って、正解クラスのスコアが高くなるように学習します。

softmax は次のように表されます。

$$
p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}
$$

正解クラスの確率が高いほど損失は小さくなります。


In [ ]:
# ===== Step 3: モデル定義 =====
class SimpleNN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 2),
        )

    def forward(self, x):
        return self.layers(x)

model = SimpleNN(input_size=X_train_tensor.shape[1]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print(model)


In [ ]:
# ===== 追加演習: 1件だけモデルに通して出力の形を見る =====
model.eval()
with torch.no_grad():
    sample_output = model(X_train_tensor[:1].to(device))
    sample_prob = torch.softmax(sample_output, dim=1)

print('出力スコア:', sample_output.cpu().numpy())
print('softmax後の確率:', sample_prob.cpu().numpy())
print('予測クラス:', sample_prob.argmax(dim=1).item())


### Step 4: モデルを学習する
epoch ごとに訓練データで重みを更新し、訓練データとテストデータの loss と accuracy を記録します。


#### 解説: エポックと過学習

エポックとは、学習データ全体を何回使って学習したかを表す単位です。エポックを増やすと、訓練データへの当てはまりは良くなりやすいですが、テストデータでの性能が必ず良くなるとは限りません。

過学習の典型的な見え方:

- train loss は下がり続ける
- test loss は途中から下がらない、または上がる
- train accuracy と test accuracy の差が大きくなる

学習曲線は、モデルが「学べているか」「覚えすぎていないか」を見るための図です。


In [ ]:
# ===== Step 4: 学習 =====
def evaluate(loader):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_X, batch_y in loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            total_loss += loss.item() * len(batch_X)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch_y.cpu().numpy())

    return total_loss / len(loader.dataset), accuracy_score(all_labels, all_preds)

history = []
epochs = 30

for epoch in range(epochs):
    model.train()
    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

    train_loss, train_acc = evaluate(train_loader)
    test_loss, test_acc = evaluate(test_loader)
    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'test_loss': test_loss,
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
    })

history_df = pd.DataFrame(history)
display(history_df.tail())


### Step 5: 学習曲線を確認する
loss が下がっているか、訓練データとテストデータの accuracy に差が出ていないかを確認します。差が大きくなる場合は、過学習の兆候かもしれません。


In [ ]:
# ===== Step 5: 学習曲線 =====
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train')
axes[0].plot(history_df['epoch'], history_df['test_loss'], label='test')
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('loss')
axes[0].set_title('Loss の推移')
axes[0].legend()

axes[1].plot(history_df['epoch'], history_df['train_accuracy'], label='train')
axes[1].plot(history_df['epoch'], history_df['test_accuracy'], label='test')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('accuracy')
axes[1].set_title('Accuracy の推移')
axes[1].legend()

plt.tight_layout()
plt.show()


## 発展演習

時間があれば、次を試してください。

1. `epochs` を 10、30、80 に変えて学習曲線を比較する。
2. 隠れ層のユニット数 `16` や `8` を変える。
3. `lr=0.01` を `0.001` や `0.05` に変える。
4. train と test の差が大きくなったら、どの時点から過学習っぽいか説明する。

モデルを複雑にすれば必ず良くなるわけではありません。結果を図で確認しましょう。


## 振り返り
- 入力層、隠れ層、出力層は、このモデルのどこにありましたか。
- epoch が進むと loss はどのように変化しましたか。
- 訓練データとテストデータの accuracy に差はありましたか。
- CNN はニューラルネットワークの一種ですが、画像データ向けに何が追加されると思いますか。
